# ResNet-18 — RSNA 2024 Lumbar Spine

In [1]:
import torch
print('torch:', torch.__version__, '| cuda:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('CHUA BAT GPU: Settings -> Accelerator -> GPU roi chay lai')

try:
    import pydicom
except ImportError:
    get_ipython().system('pip install -q pydicom')
    import pydicom
import pandas as pd, numpy as np, os
print('pydicom ok')

torch: 2.11.0+cu128 | cuda: True
GPU: Tesla T4
pydicom ok


In [2]:
import torch.nn as nn


class BasicBlock(nn.Module):
    expansion = 1

    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, 3, stride, 1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, 3, 1, 1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)
        self.shortcut = nn.Sequential()
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, 1, stride, bias=False),
                nn.BatchNorm2d(out_channels))

    def forward(self, x):
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        return self.relu(out + self.shortcut(x))


class ResNet(nn.Module):
    def __init__(self, in_channels=1, num_classes=3):
        super().__init__()
        self.in_channels = 64
        self.stem = nn.Sequential(
            nn.Conv2d(in_channels, 64, 7, 2, 3, bias=False),
            nn.BatchNorm2d(64), nn.ReLU(inplace=True),
            nn.MaxPool2d(3, 2, 1))
        self.layer1 = self._layer(BasicBlock, 64, 2, 1)
        self.layer2 = self._layer(BasicBlock, 128, 2, 2)
        self.layer3 = self._layer(BasicBlock, 256, 2, 2)
        self.layer4 = self._layer(BasicBlock, 512, 2, 2)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Linear(512, num_classes)

    def _layer(self, block, out_c, n, stride):
        layers = [block(self.in_channels, out_c, stride)]
        self.in_channels = out_c
        layers += [block(self.in_channels, out_c, 1) for _ in range(1, n)]
        return nn.Sequential(*layers)

    def forward(self, x):
        x = self.stem(x)
        for l in (self.layer1, self.layer2, self.layer3, self.layer4):
            x = l(x)
        return self.fc(torch.flatten(self.pool(x), 1))


model = ResNet().cuda() if torch.cuda.is_available() else ResNet()
n_params = sum(p.numel() for p in model.parameters())
print(f'ResNet-18 params: {n_params:,}')  # ~11.2M

ResNet-18 params: 11,171,779


In [6]:
import os
print(os.listdir('/kaggle/input/competitions'))

['rsna-2024-lumbar-spine-degenerative-classification']


In [17]:
from torch.utils.data import Dataset, DataLoader

ROOT = '/kaggle/input/competitions/rsna-2024-lumbar-spine-degenerative-classification'
LABEL2IDX = {'Normal/Mild': 0, 'Moderate': 1, 'Severe': 2}
NAMES = ['Normal/Mild', 'Moderate', 'Severe']
CROP = 128
MAX_STUDIES = None  


def col_of(cond, level):
    return cond.lower().replace(' ', '_') + '_' + level.lower().replace('/', '_')


labels = pd.read_csv(f'{ROOT}/train.csv').set_index('study_id')
coords = pd.read_csv(f'{ROOT}/train_label_coordinates.csv')
if MAX_STUDIES:
    keep = labels.index.tolist()[:MAX_STUDIES]
    coords = coords[coords.study_id.isin(keep)]

rows = []
for r in coords.itertuples():
    col = col_of(r.condition, r.level)
    sev = labels.at[r.study_id, col] if col in labels.columns else None
    if not isinstance(sev, str) or sev not in LABEL2IDX:
        continue
    p = f"{ROOT}/train_images/{r.study_id}/{r.series_id}/{int(r.instance_number)}.dcm"
    if os.path.exists(p):
        rows.append((p, float(r.x), float(r.y), LABEL2IDX[sev], r.study_id))
df = pd.DataFrame(rows, columns=['path', 'x', 'y', 'label', 'study'])
print(f'{len(df)} crops tu {df.study.nunique()} studies')
print(df.label.value_counts(normalize=True).sort_index().to_dict())


def load_crop(path, x, y):
    img = pydicom.dcmread(path).pixel_array.astype(np.float32)
    h, w = img.shape
    cx = min(max(int(round(x)), CROP // 2), w - CROP // 2)
    cy = min(max(int(round(y)), CROP // 2), h - CROP // 2)
    c = img[cy - CROP // 2:cy + CROP // 2, cx - CROP // 2:cx + CROP // 2]
    c = (c - c.min()) / (c.max() - c.min() + 1e-6)
    return torch.from_numpy(c).unsqueeze(0)


class CachedCrops(Dataset):
    def __init__(self, positions):
        self.positions = positions

    def __len__(self):
        return len(self.positions)

    def __getitem__(self, i):
        return _CACHE[self.positions[i]]


print('preload crops vao RAM (1 lan duy nhat, doc song song)...')
from concurrent.futures import ThreadPoolExecutor


def _load_one(i):
    r = df.iloc[i]
    return load_crop(r['path'], r['x'], r['y']), int(r['label'])


with ThreadPoolExecutor(max_workers=8) as _ex:
    _CACHE = list(_ex.map(_load_one, range(len(df))))
print(f"cache xong {len(_CACHE)} crops (~{len(_CACHE) * 128 * 128 * 4 // 1024 // 1024}MB RAM)")

48657 crops tu 1974 studies
{0: 0.7732905851162217, 1: 0.16338861828719403, 2: 0.06332079659658425}
preload crops vao RAM (1 lan duy nhat, doc song song)...
cache xong 48657 crops (~3041MB RAM)


In [18]:
# 3. Train tren GPU: split theo study, weighted loss, luu model tot nhat
import time

EPOCHS, BATCH, LR = 8, 64, 1e-3
device = 'cuda' if torch.cuda.is_available() else 'cpu'
studies = sorted(df.study.unique())
cut = int(len(studies) * 0.85)
tr = df[df.study.isin(studies[:cut])]
va = df[df.study.isin(studies[cut:])]
tr_loader = DataLoader(CachedCrops(tr.index.tolist()), BATCH, shuffle=True, num_workers=0, pin_memory=True)
va_loader = DataLoader(CachedCrops(va.index.tolist()), BATCH, num_workers=0, pin_memory=True)
print(f'train {len(tr)} / val {len(va)} crops')

freq = tr.label.value_counts(normalize=True).sort_index()
w = torch.tensor([(1.0 / freq[i]) for i in range(3)], dtype=torch.float32)
criterion = nn.CrossEntropyLoss(weight=(w / w.mean()).to(device))
opt = torch.optim.Adam(model.parameters(), lr=LR)
from torch.amp import autocast, GradScaler
scaler = GradScaler('cuda') if device == 'cuda' else None

best, t0 = 0, time.time()
for ep in range(EPOCHS):
    model.train()
    tl = tc = tn = 0
    for imgs, lab in tr_loader:
        imgs, lab = imgs.to(device), lab.to(device)
        opt.zero_grad()
        with autocast('cuda', enabled=scaler is not None):
            out = model(imgs)
            loss = criterion(out, lab)
        if scaler:
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
        else:
            loss.backward()
            opt.step()
        tl += loss.item() * len(imgs)
        tc += (out.argmax(1) == lab).sum().item()
        tn += len(imgs)
    model.eval()
    vl = vc = vn = 0
    with torch.no_grad():
        for imgs, lab in va_loader:
            imgs, lab = imgs.to(device), lab.to(device)
            out = model(imgs)
            vl += criterion(out, lab).item() * len(imgs)
            vc += (out.argmax(1) == lab).sum().item()
            vn += len(imgs)
    vacc = vc / vn
    print(f'Epoch {ep+1}/{EPOCHS} - train loss {tl/tn:.4f} acc {tc/tn:.2%} | val loss {vl/vn:.4f} acc {vacc:.2%}')
    if vacc > best:
        best = vacc
        torch.save(model.state_dict(), '/kaggle/working/resnet18_best.pth')

print(f'XONG {time.time()-t0:.0f}s | best val acc {best:.2%} | file: /kaggle/working/resnet18_best.pth')

train 41339 / val 7318 crops
Epoch 1/8 - train loss 0.6588 acc 75.67% | val loss 0.7511 acc 71.65%
Epoch 2/8 - train loss 0.5688 acc 77.97% | val loss 0.5974 acc 77.99%
Epoch 3/8 - train loss 0.5159 acc 79.49% | val loss 0.5973 acc 72.81%
Epoch 4/8 - train loss 0.4641 acc 80.85% | val loss 0.5947 acc 77.96%
Epoch 5/8 - train loss 0.4080 acc 82.45% | val loss 0.6467 acc 79.27%
Epoch 6/8 - train loss 0.3489 acc 84.40% | val loss 0.6812 acc 79.98%
Epoch 7/8 - train loss 0.2809 acc 86.78% | val loss 0.8705 acc 74.02%
Epoch 8/8 - train loss 0.2205 acc 89.00% | val loss 0.9642 acc 79.15%
XONG 210s | best val acc 79.98% | file: /kaggle/working/resnet18_best.pth


In [19]:
print(f'Model: ResNet-18 ({n_params:,} params) | input: crop {CROP}x{CROP} quanh annotation')
print(f'Data: {len(df)} crops / {df.study.nunique()} studies | train {len(tr)} / val {len(va)}')
print(f'Best val acc: {best:.2%} | weights: /kaggle/working/resnet18_best.pth')

Model: ResNet-18 (11,171,779 params) | input: crop 128x128 quanh annotation
Data: 48657 crops / 1974 studies | train 41339 / val 7318
Best val acc: 79.98% | weights: /kaggle/working/resnet18_best.pth


In [20]:
model.load_state_dict(torch.load('/kaggle/working/resnet18_best.pth'))
model.eval()
cm = torch.zeros(3, 3, dtype=torch.long)
with torch.no_grad():
    for imgs, lab in va_loader:
        pred = model(imgs.to(device)).argmax(1).cpu()
        for t, p in zip(lab, pred):
            cm[t, p] += 1
print(cm.numpy())
rec = cm.diag().float() / cm.sum(1).clamp_min(1)
for i, n in enumerate(NAMES):
    print(f'{n}: recall {rec[i]:.2%}')
print(f'balanced acc: {rec.mean():.2%}')

[[4817  872   60]
 [ 194  697  239]
 [  12   88  339]]
Normal/Mild: recall 83.79%
Moderate: recall 61.68%
Severe: recall 77.22%
balanced acc: 74.23%
